06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [14]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io

cloth_info = pd.read_csv('data/clothDataset_1_.csv')

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (30, 131)
cloth_info: 
    frame        x0        y0        z0        vx0        vy0       vz0  \
0       1 -0.002500 -0.005000  0.000000   0.000000   0.000000  0.000000   
1       2 -0.250000 -0.500000  0.000000 -12.375010 -24.750010  0.000000   
2       3 -0.249982 -0.499981  0.000000   0.000893   0.000939  0.000000   
3       4 -0.250010 -0.500045  0.000000  -0.001390  -0.003207  0.000000   
4       5 -0.250033 -0.499865  0.000000  -0.001153   0.009009  0.000000   
5       6 -0.249931 -0.499897  0.000000   0.005092  -0.001603  0.000000   
6       7 -0.250036 -0.500017  0.000000  -0.005226  -0.005975  0.000000   
7       8 -0.249945 -0.500060  0.000000   0.004528  -0.002176  0.000000   
8       9 -0.249992 -0.499946  0.000000  -0.002334   0.005707  0.000000   
9      10 -0.250081 -0.499962  0.000000  -0.004445  -0.000790  0.000000   
10     11 -0.250029 -0.499865  0.000000   0.002611   0.004849  0.000000   
11     12 -0.250076 -0.499875  0.000000  -0.002348  -0.0004

In [18]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=10):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        if isinstance(csv_data, str) and "frame,x0" in csv_data:
            self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        else:
            self.data = pd.read_csv(csv_data)
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()

        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'dx', 'dy', 'dz']

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)

    def __getitem__(self, idx):
        # Fetch the row corresponding to the frame
        row = self.data.iloc[idx]
        
        frame_data = []
        
        # Loop through each vertex and grab its 13 specific columns
        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)
            
        # Convert the list of arrays into a PyTorch tensor
        # Final shape: (num_vertices, num_features) -> (10, 13)
        tensor_data = torch.tensor(np.array(frame_data))
        
        # Extract the frame number as an integer label/identifier
        frame_id = torch.tensor(row['frame'], dtype=torch.int32)
        
        return tensor_data, frame_id

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset('data/clothDataset_1_.csv')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}") # Expected: [4, 10, 13]
    print(batch_data)
    break

Batch Shape: torch.Size([4, 10, 13])
tensor([[[-2.5003e-01, -4.9994e-01,  0.0000e+00, -5.3644e-05,  1.3575e-03,
           0.0000e+00, -5.0000e-01, -2.4990e-01,  0.0000e+00,  3.1084e-03,
          -6.4783e-01, -7.6179e-01,  0.0000e+00],
         [ 4.1939e-03,  0.0000e+00, -5.0007e-01, -4.9992e-01,  0.0000e+00,
          -4.1872e-03,  4.5538e-03,  0.0000e+00, -2.4997e-01, -2.4996e-01,
          -7.7130e-01, -6.3647e-01,  0.0000e+00],
         [ 0.0000e+00,  1.7852e-03,  3.1367e-04,  0.0000e+00, -6.8012e-05,
          -5.0002e-01,  0.0000e+00, -1.0953e-03, -3.1918e-03,  0.0000e+00,
          -7.1195e-01, -7.0223e-01,  0.0000e+00],
         [-4.9996e-01,  1.0261e-04,  0.0000e+00,  1.7077e-03,  3.4414e-03,
           0.0000e+00, -1.7103e-05, -2.5001e-01,  0.0000e+00,  3.7087e-03,
          -7.1285e-01, -7.0132e-01,  0.0000e+00],
         [-4.3318e-03,  0.0000e+00,  2.4999e-01, -5.0012e-01,  0.0000e+00,
          -7.2196e-04, -1.0037e-02,  0.0000e+00, -2.4997e-01,  6.8977e-05,
          -5.

In [ ]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

In [ ]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
